# npe bootstrap

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import time
from pathlib import Path
import torch
from scipy.stats import gaussian_kde, ks_2samp
from scipy.special import rel_entr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform
import warnings
import functools
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# raw data
cache_dir = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir   = PROJECT / "analogues/tng_for_comparison_v3"


with open(cache_dir / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(cache_dir / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)

keep_idx    = np.load(tng_dir / "keep_idx.npy")
tng_pairs_i = np.load(tng_dir / "pairs_i.npy")
tng_pairs_j = np.load(tng_dir / "pairs_j.npy")
tng_gr_sample   = np.load(tng_dir / "g_r.npy")
tng_met_sample  = np.load(tng_dir / "metallicity.npy")
tng_separation  = np.load(tng_dir / "separation.npy")
tng_vr          = np.load(tng_dir / "vr.npy")
tng_vt          = np.load(tng_dir / "vt.npy")
tng_log_m200c   = np.load(tng_dir / "log_m200c.npy")
tng_log_mass_ratio = np.load(tng_dir / "log_mass_ratio.npy")
tng_same_host   = np.load(tng_dir / "have_same_host.npy").astype(bool)

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_sigma_j = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_j_global])
tng_kappa_i = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_kappa_j = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_j_global])

# feature blocks
BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio", "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio", "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}
TARGET = "log_m200c"

# we only use quantities which are calculated from position and velocities for inference.
# seperation 761 +- 11 kpc Li et al. 2021
# vr,vt van der Marel et al. 2012
# vt contested Salomon et al. 2021 give vt = 82.4 +-31.2 km/s. We run both
# log_mass_ratio: ratio of total bound masses not stellar. Sawala et al. 2023. M_mw = 1.1 +- 0.2
# M_31 = 1.5 +- 0.5 x 10^12 Msun.
# One thing to note though boundsubhalo/totalmass is not equal to observational m200 and does not cancel exactly in a ratio; this is a proxy assumption
LG_OBS, LG_UNC = lg_observation("vdM2012")

LG_SAFE_BLOCKS = ["core_dynamics"]
BOOTSTRAP_BLOCKS = [b for b in ["core_dynamics", "core_morphology"]
                    if b in LG_SAFE_BLOCKS]

def pair_features_flamingo(sample, pairs, met_transform):
    pi, pj = pairs["i"], pairs["j"]

    gr_i    = sample["g_r"][pi];          gr_j    = sample["g_r"][pj]
    sigma_i = sample["vel_disp"][pi];     sigma_j = sample["vel_disp"][pj]
    kap_i   = sample["kappa_rot"][pi];    kap_j   = sample["kappa_rot"][pj]
    met_i   = met_transform["flamingo_met_dex"][pi]
    met_j   = met_transform["flamingo_met_dex"][pj]

    df = pd.DataFrame({
        "gr_mean":        (gr_i    + gr_j)    / 2,
        "gr_diff":        np.abs(gr_i    - gr_j),
        "met_mean":       (met_i   + met_j)   / 2,
        "met_diff":       np.abs(met_i   - met_j),
        "sigma_mean":     (sigma_i + sigma_j) / 2,
        "sigma_diff":     np.abs(sigma_i - sigma_j),
        "kappa_mean":     (kap_i   + kap_j)   / 2,
        "kappa_diff":     np.abs(kap_i   - kap_j),
        "separation":     pairs["separation_kpc"],
        "vr":             pairs["vr"],
        "vt":             pairs["vt"],
        "log_mass_ratio": pairs["log_mass_ratio"],
        "log_m200c":      pairs["log_tot_virial_mass"],
    })
    return df

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):

    gr_i  = tng_gr_sample[tng_pairs_i];  gr_j  = tng_gr_sample[tng_pairs_j]

    # metallicity: log10(Z/Zsun) z-scored with TNG's own scaler
    met_raw_i = tng_met_sample[tng_pairs_i]
    met_raw_j = tng_met_sample[tng_pairs_j]
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]

    df = pd.DataFrame({
        "gr_mean":    (gr_i        + gr_j)        / 2,
        "gr_diff":    np.abs(gr_i        - gr_j),
        "met_mean":   (met_i       + met_j)        / 2,
        "met_diff":   np.abs(met_i       - met_j),
        "sigma_mean": (tng_sigma_i + tng_sigma_j)  / 2,
        "sigma_diff": np.abs(tng_sigma_i - tng_sigma_j),
        "kappa_mean": (tng_kappa_i + tng_kappa_j)  / 2,
        "kappa_diff": np.abs(tng_kappa_i - tng_kappa_j),
        "separation": tng_separation,
        "vr":         tng_vr,
        "vt":         tng_vt,
        "log_mass_ratio": tng_log_mass_ratio,
        "log_m200c":  tng_log_m200c,               # target for now will add some more later
    })
    return df


flam_df = pair_features_flamingo(sample, pairs, met_transform)
tng_df  = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)
flam_df["same_host"] = pairs["have_same_host"].astype(bool)
tng_df["same_host"]  = tng_same_host

ALL_FEATURES = BLOCKS["all_features"]
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
tng_df_clean  = tng_df.dropna(subset=ALL_FEATURES + [TARGET])

# previously trained FLAMINGO and TNG posteriors
results_dir = configure_run(PROJECT)
results_dir.mkdir(parents=True, exist_ok=True)

def load_checkpoint_any_device(path, device):
    """Loads a pickle containing nested CUDA tensors safely on CPU-only
    machines, by patching torch.load's default map_location for the
    duration of the load (needed because pickle's internal tensor
    deserialization calls torch.load directly, bypassing any map_location
    passed to the outer call)."""
    original_torch_load = torch.load
    torch.load = functools.partial(original_torch_load, map_location=device)
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    finally:
        torch.load = original_torch_load  # always restore, even if load fails
    return obj

with open(LEGACY_RESULTS_DIR / "optuna_search_results.pkl", "rb") as f:
    best_params = pickle.load(f)["best_params"]
print("Loaded best_params:", best_params)

flam_ckpt = load_checkpoint_any_device(results_dir / "flamingo_npe_checkpoint.pkl", device)
flam_samples = flam_ckpt["samples"]

tng_ckpt = load_checkpoint_any_device(results_dir / "tng_npe_checkpoint.pkl", device)
tng_samples = tng_ckpt["samples"]

with open(results_dir / "js_divergence_results.pkl", "rb") as f:
    js_results = pickle.load(f)

def js_divergence(p_samples, q_samples, xs, base=2, bw_method=0.15):
    p = gaussian_kde(p_samples, bw_method=bw_method)(xs)
    q = gaussian_kde(q_samples, bw_method=bw_method)(xs)
    eps = 1e-12
    p = np.clip(p / p.sum(), eps, None); p /= p.sum()
    q = np.clip(q / q.sum(), eps, None); q /= q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(rel_entr(p, m)) / np.log(base)
    kl_qm = np.sum(rel_entr(q, m)) / np.log(base)
    return 0.5 * kl_pm + 0.5 * kl_qm

print(f"FLAMINGO pool: {len(flam_df_clean):,}   TNG: {len(tng_df_clean):,}")
print(f"Loaded {len(flam_samples)} FLAMINGO combos, {len(tng_samples)} TNG blocks, "
      f"JSD results for {len(js_results)} blocks.")

prior = BoxUniform(low=torch.tensor([11.5], device=device), high=torch.tensor([14.5], device=device))

In [ ]:
from scipy.stats import ks_2samp

def population_bootstrap(flam_pool, n_target, n_repeats=500, seed=0):
    """Null: how large a KS statistic does a sample of TNG's size, drawn from
    FLAMINGO's own population, produce against the rest of FLAMINGO?"""
    rng = np.random.default_rng(seed)
    ks_null = np.empty(n_repeats)
    for i in range(n_repeats):
        idx  = rng.permutation(len(flam_pool))
        sub  = flam_pool[idx[:n_target]]
        rest = flam_pool[idx[n_target:]]
        ks_null[i] = ks_2samp(sub, rest).statistic
    return ks_null

# theta-marginal only, so this is block-independent: compute once.
flam_theta = flam_df_clean[TARGET].dropna().values
tng_theta  = tng_df_clean[TARGET].dropna().values

ks_null = population_bootstrap(flam_theta, len(tng_theta))
ks_real = ks_2samp(flam_theta, tng_theta).statistic
p_val = (1 + np.count_nonzero(ks_null >= ks_real)) / (len(ks_null) + 1)

pop_results = {"theta_marginal": {
    "ks_null_mean": ks_null.mean(), "ks_null_std": ks_null.std(),
    "ks_real": ks_real, "p_value": p_val, "n_tng": len(tng_theta),
}}
print(f"theta-marginal   KS(null): {ks_null.mean():.3f}±{ks_null.std():.3f}  "
      f"KS(real): {ks_real:.3f}  p={p_val:.4f}  (n_tng={len(tng_theta)})")

with open(results_dir / "population_bootstrap_results.pkl", "wb") as f:
    pickle.dump(pop_results, f)

In [ ]:
BOOTSTRAP_CHECKPOINT = results_dir / "subsampling_bootstrap_checkpoint.pkl"
N_REPEATS = 200

bootstrap_null = {}
if BOOTSTRAP_CHECKPOINT.exists():
    with open(BOOTSTRAP_CHECKPOINT, "rb") as f:
        bootstrap_null = pickle.load(f)
    if bootstrap_null:
        print(f"Resuming: {sum(len(v) for v in bootstrap_null.values())} repeat(s) already done.")

xs = np.linspace(11.5, 14.5, 500)
run_times = []

for block_name in BOOTSTRAP_BLOCKS:
    feats = BLOCKS[block_name]
    valid_flam = flam_df_clean[feats + [TARGET]].dropna()
    valid_tng  = tng_df_clean[feats + [TARGET]].dropna()
    n_target = len(valid_tng)

    if block_name not in bootstrap_null:
        bootstrap_null[block_name] = []
    done = len(bootstrap_null[block_name])

    for rep in range(done, N_REPEATS):
        t0 = time.time()
        seed = 2000 + rep
        sample_df = valid_flam.sample(n=n_target, random_state=seed)

        X_raw = sample_df[feats].values.astype(np.float32)
        y_raw = sample_df[TARGET].values.astype(np.float32)
        scaler = StandardScaler()
        X_sc   = scaler.fit_transform(X_raw)

        theta = torch.tensor(y_raw.reshape(-1, 1), device=device)
        x     = torch.tensor(X_sc, device=device)

        torch.manual_seed(seed)
        np.random.seed(seed)

        density_builder = posterior_nn(
            model="maf",
            hidden_features=best_params["hidden_features"],
            num_transforms=best_params["num_transforms"],
        )
        inference = SNPE(prior=prior, density_estimator=density_builder, device=device)
        inference.append_simulations(theta, x)
        density_est = inference.train(
            training_batch_size=best_params["batch_size"],
            learning_rate=best_params["learning_rate"],
            max_num_epochs=200,
            stop_after_epochs=20,
            show_train_summary=False,
        )
        posterior = inference.build_posterior(density_est)

        lg_vec = np.array([LG_OBS[f] for f in feats], dtype=np.float32).reshape(1, -1)
        x_obs  = torch.tensor(scaler.transform(lg_to_peculiar(lg_vec, feats, H0_BY_SIM["flamingo"])), device=device)
        sub_samples = posterior.sample((10_000,), x=x_obs).cpu().numpy().squeeze()

        jsd_vs_full = js_divergence(sub_samples, flam_samples[f"{block_name}__full"], xs)
        jsd_vs_tng  = js_divergence(sub_samples, tng_samples[block_name], xs)

        bootstrap_null[block_name].append({
            "seed": seed,
            "samples": sub_samples[:2000],
            "median": float(np.median(sub_samples)),
            "mean": float(np.mean(sub_samples)),
            "std": float(np.std(sub_samples)),
            "ci68": np.percentile(sub_samples, [16, 84]).tolist(),
            "ci95": np.percentile(sub_samples, [2.5, 97.5]).tolist(),
            "jsd_vs_full": jsd_vs_full,
            "jsd_vs_tng": jsd_vs_tng,
            "epochs_trained": inference.summary["epochs_trained"][-1],
            "best_validation_loss": inference.summary["best_validation_loss"][-1],
        })

        elapsed = time.time() - t0
        run_times.append(elapsed)
        avg = sum(run_times) / len(run_times)
        remaining = sum(N_REPEATS - len(bootstrap_null.get(b, [])) for b in BOOTSTRAP_BLOCKS)

        print(f"{block_name}  rep {rep+1}/{N_REPEATS}  JSD(vs full)={jsd_vs_full:.4f}  "
              f"({elapsed/60:.1f} min, ETA {avg*remaining/60:.0f} min)")

        with open(BOOTSTRAP_CHECKPOINT, "wb") as f:
            pickle.dump(bootstrap_null, f)

print("\nBootstrap done.")

In [ ]:
N_TRAINING_NOISE_REPS = 10
training_noise_results = {}

for block_name in BOOTSTRAP_BLOCKS:
    feats = BLOCKS[block_name]
    valid_flam = flam_df_clean[feats + [TARGET]].dropna()
    n_target = len(tng_df_clean[feats + [TARGET]].dropna())

    fixed_sample_df = valid_flam.sample(n=n_target, random_state=9999)
    X_raw = fixed_sample_df[feats].values.astype(np.float32)
    y_raw = fixed_sample_df[TARGET].values.astype(np.float32)
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)

    lg_vec = np.array([LG_OBS[f] for f in feats], dtype=np.float32).reshape(1, -1)
    x_obs  = torch.tensor(scaler.transform(lg_to_peculiar(lg_vec, feats, H0_BY_SIM["flamingo"])), device=device)

    medians = []
    for seed in range(N_TRAINING_NOISE_REPS):
        torch.manual_seed(seed)
        np.random.seed(seed)

        theta = torch.tensor(y_raw.reshape(-1, 1), device=device)
        x     = torch.tensor(X_sc, device=device)

        density_builder = posterior_nn(
            model="maf",
            hidden_features=best_params["hidden_features"],
            num_transforms=best_params["num_transforms"],
        )
        inference = SNPE(prior=prior, density_estimator=density_builder, device=device)
        inference.append_simulations(theta, x)
        density_est = inference.train(
            training_batch_size=best_params["batch_size"],
            learning_rate=best_params["learning_rate"],
            max_num_epochs=200, stop_after_epochs=20, show_train_summary=False,
        )
        posterior = inference.build_posterior(density_est)
        samp = posterior.sample((10_000,), x=x_obs).cpu().numpy().squeeze()
        medians.append(np.median(samp))

    training_noise_results[block_name] = np.array(medians)
    print(f"{block_name}: training-noise-only spread  "
          f"std={np.std(medians):.4f}  range={np.ptp(medians):.4f}")

with open(results_dir / "training_noise_diagnostic.pkl", "wb") as f:
    pickle.dump(training_noise_results, f)

In [ ]:
def summarize(arr):
    return {"median": np.median(arr),
            "ci68": (np.percentile(arr, 16), np.percentile(arr, 84)),
            "ci95": (np.percentile(arr, 2.5), np.percentile(arr, 97.5))}

fig, axes = plt.subplots(1, len(BOOTSTRAP_BLOCKS), figsize=(7*len(BOOTSTRAP_BLOCKS), 5))
if len(BOOTSTRAP_BLOCKS) == 1:
    axes = [axes]

for ax, block_name in zip(axes, BOOTSTRAP_BLOCKS):
    reps = bootstrap_null[block_name]
    null_vs_full = np.array([r["jsd_vs_full"] for r in reps])

    rng = np.random.default_rng(0)
    idx = rng.permutation(len(reps))
    pairs_idx = list(zip(idx[::2], idx[1::2]))
    null_pairwise = np.array([js_divergence(reps[i]["samples"], reps[j]["samples"], xs)
                                for i, j in pairs_idx])

    real_jsd = js_results[block_name]["jsd_full"]
    s1, s2 = summarize(null_vs_full), summarize(null_pairwise)
    p1 = (1 + np.count_nonzero(null_vs_full >= real_jsd)) / (len(null_vs_full) + 1)
    p2 = (1 + np.count_nonzero(null_pairwise >= real_jsd)) / (len(null_pairwise) + 1)

    print(f"\n{block_name}")
    print(f"  null (subsample vs FLAM-full): median={s1['median']:.4f}  "
          f"95% CI=({s1['ci95'][0]:.4f}, {s1['ci95'][1]:.4f})  p={p1:.4f}")
    print(f"  null (subsample vs subsample): median={s2['median']:.4f}  "
          f"95% CI=({s2['ci95'][0]:.4f}, {s2['ci95'][1]:.4f})  p={p2:.4f}")
    print(f"  observed FLAM(full) vs TNG:    {real_jsd:.4f}")

    ax.hist(null_vs_full, bins=30, alpha=0.5, color="tab:blue", label="null: subsample vs FLAM-full")
    ax.hist(null_pairwise, bins=30, alpha=0.5, color="tab:green", label="null: subsample vs subsample")
    ax.axvline(real_jsd, color="tab:red", lw=2, ls="--", label="observed FLAM(full) vs TNG")
    ax.set_xlabel("JSD"); ax.set_title(block_name); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Block':<20} {'Bootstrap null std':>20} {'Training-noise-only std':>26} {'Ratio':>8}")

noise_comparison = {}
for block_name in BOOTSTRAP_BLOCKS:
    bootstrap_medians = np.array([r["median"] for r in bootstrap_null[block_name]])
    bootstrap_std = bootstrap_medians.std()
    training_std  = training_noise_results[block_name].std()
    ratio = training_std / bootstrap_std if bootstrap_std > 0 else np.nan

    noise_comparison[block_name] = {"bootstrap_std": bootstrap_std, "training_std": training_std, "ratio": ratio}
    print(f"{block_name:<20} {bootstrap_std:>20.4f} {training_std:>26.4f} {ratio:>8.2f}")

with open(results_dir / "noise_comparison.pkl", "wb") as f:
    pickle.dump(noise_comparison, f)

In [ ]:
print("Subsampling Final Summary")

for block_name in BOOTSTRAP_BLOCKS:
    reps = bootstrap_null[block_name]
    null_vs_full = np.array([r["jsd_vs_full"] for r in reps])
    real_jsd = js_results[block_name]["jsd_full"]
    p_val = (1 + np.count_nonzero(null_vs_full >= real_jsd)) / (len(null_vs_full) + 1)

    print(f"\n{block_name}")
    print(f"  N repeats:            {len(reps)}")
    print(f"  Null median JSD:      {np.median(null_vs_full):.4f}")
    print(f"  Null 95% CI:          ({np.percentile(null_vs_full,2.5):.4f}, "
          f"{np.percentile(null_vs_full,97.5):.4f})")
    print(f"  Observed JSD (real):  {real_jsd:.4f}")
    print(f"  p-value:              {p_val:.4f}")
    print(f"  Training-noise std:   {noise_comparison[block_name]['training_std']:.4f} "
          f"(vs bootstrap null std {noise_comparison[block_name]['bootstrap_std']:.4f})")
    verdict = "SIGNIFICANT -- real divergence exceeds sampling noise" if p_val < 0.05 else \
              "NOT significant -- consistent with sampling noise alone"
    print(f"  Verdict:              {verdict}")

print(f"\nNote: covers {BOOTSTRAP_BLOCKS} only.")